# Evaluation of seasonal distribution of models on CORDEX-ML_BENCH datasets

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.cordex_ml_default_params import *

In [ ]:
import functools
import math
import string

import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis.data import prep_eval_data
from mlde_analysis import plot_map
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, stat_bias, plot_freq_density, rms_mean_bias, rms_std_bias, xr_hist, hist_dist, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS
from mlde_analysis.wet_dry import threshold_exceeded_prop_stats, threshold_exceeded_prop, threshold_exceeded_prop_error, threshold_exceeded_prop_change, plot_threshold_exceedence_errors, THRESHOLDS
from mlde_utils import cp_model_rotated_pole
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Figures: Seasonal distribution

* Frequency Density Histogram of rainfall intensities
* Maps of Mean bias ($\frac{\mu_{sample}-\mu_{CPM}}{\mu_{CPM}}$) over all samples, time and ensemble members
* Std Dev Bias $\frac{\sigma_{sample}}{\sigma_{CPM}}$ over all samples, time and ensemble members

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    metrics_ds = VAR_DAS[var].groupby("time.season").map(lambda season_ds:compute_metrics(season_ds[f"pred_{var}"], season_ds[f"target_{var}"], thresholds=DIST_THRESHOLDS[var]))

    pretty_table(metrics_ds, round=4, dim_order=["season", "model"])
    
    for season, season_ds in VAR_DAS[var].groupby("time.season"):
        IPython.display.display_markdown(f"#### {season}", raw=True)
        hist_das = season_ds[f"pred_{var}"]
        target_da = season_ds[f"target_{var}"]
        normalize=(var == "pr")
        mean_biases = season_ds[f"pred_{var}"].groupby("model").map(mean_bias, target_da=target_da, normalize=normalize)
        std_biases = season_ds[f"pred_{var}"].groupby("model").map(std_bias, target_da=target_da, normalize=normalize)
        
        f_q999_bias = functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999))
        q999_biases = season_ds[f"pred_{var}"].groupby("model").map(f_q999_bias, target_da=target_da, normalize=normalize)
        
        bias_kwargs = {"style": f"{var}Bias"}
        for fd_kwargs in [{"yscale": "log"}, {"yscale": "linear"}]:
            if var == "pr" and fd_kwargs["yscale"] == "linear":
                continue
            fig = plt.figure(layout="constrained", figsize=(5.5, 6.5))
            axd = plot_distribution_figure(
                fig,
                hist_das,
                target_da,
                mean_biases,
                std_biases,
                q999_biases,
                MODELLABEL2SPEC, 
                hrange=VAR_RANGES[var],
                fd_kwargs=fd_kwargs,
                bias_kwargs=bias_kwargs,
            )
            if var == "relhum150cm":
                axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
            
        plt.show()

## Seasonal QQ plots

In [ ]:
quantile_dims=["ensemble_member", "T", "X", "Y"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    fig = plt.figure(layout='constrained', figsize=(5.5, 5.5))
    axd = fig.subplot_mosaic(np.array(["DJF", "MAM", "JJA", "SON"]).reshape(2,2))

    for season, season_ds in VAR_DAS[var].groupby("time.season"):
        season_target_da = season_ds[f"target_{var}"]

        quantiles = reasonable_quantiles(season_target_da)
        season_target_quantiles = season_target_da.cf.quantile(quantiles, dim=quantile_dims).rename("target_q")

        season_pred_da = season_ds[f"pred_{var}"]
        season_pred_quantiles = season_pred_da.cf.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        xlabel = f"Target \n{xr.plot.utils.label_from_attrs(da=season_target_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=season_pred_da)}"

        qq_plot(axd[season], season_target_quantiles, season_pred_quantiles, title=season, xlabel=xlabel, ylabel=ylabel)

    plt.show()